Extract and consolidate fastF1 data

In [ ]:
import fastf1
import pandas as pd
import os
import time
from pathlib import Path

#Project Root
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name =="notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
    
#Year to be searched
YEAR = 2026
CACHE_DIR = PROJECT_DIR / "Data" / "Cache"
OUTPUT_DIR = PROJECT_DIR / "Data" / 'Raw'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fastf1.Cache.configure(cache_dir=str(CACHE_DIR))

output_file = OUTPUT_DIR / f"result_{YEAR}.parquet"

# Avoiding re-collecting a year that has already been saved
if  output_file.exists():
    df_existente = pd.read_parquet(output_file)
    rodadas_coletadas = df_existente['Round'].unique().tolist()
    print(f"Season {YEAR} already have the folloing GP's {rodadas_coletadas}")
else:
    df_existente = pd.DataFrame()
    rodadas_coletadas = []
    print(f" Season {YEAR} collecting initiated!")

    try:
        schedule = fastf1.get_event_schedule(
            YEAR,
            include_testing=False
        )
    except fastf1.req.RateLimitExceededError:
        print("Request limit exceeded while fetching the calendar.")
        print("Wait before trying again.")
        schedule = None

    except Exception as e:
        print(f"Error fetching the calendar: {e}")
        schedule = None

    novos_resultados = []

    if schedule is not None:
        now = pd.Timestamp.now()

        for _, event in schedule.iterrows():
            round_number = int(event["RoundNumber"])
    
            if round_number == 0 or round_number in rodadas_coletadas:
                continue
            # ignore futures events
    
            if pd.Timestamp(event["EventDate"]) > now:
                continue

            print(f"Collecting GP {round_number}: {event['EventName']}")

            try:
                session = fastf1.get_session(
                    YEAR,
                    round_number,
                    "R"
                )

                session.load(
                    laps=False,
                    telemetry=False,
                    weather=False
                )

                results = session.results.copy()

                if results.empty:
                    continue


                for _, driver in results.iterrows():

                    novos_resultados.append({
                        "Season": YEAR,
                        "Round": round_number,
                        "Date": event["EventDate"],
                        "Circuit": event["Location"],
                        "Event": event["EventName"],
                        "Driver": driver["Abbreviation"],
                        "Team": driver["TeamName"],
                        "GridPosition": driver["GridPosition"],
                        "Position": driver["Position"]
                    })
            except fastf1.req.RateLimitExceededError:
                print(" Requisition limit exceeded")
                print("Saving collected date until now")
                break

            except Exception as e:
                print(f"Error GP {round_number}: {e}")

            time.sleep(1)


        if novos_resultados:
            df_novos = pd.DataFrame(novos_resultados)
            df_final = pd.concat([df_existente, df_novos], ignore_index=True)
            df_final.sort_values(by=["Round", "Position"], inplace=True)
            df_final.to_parquet(output_file, index=False)
            print(f"\nSeason {YEAR} saved!")
            print(f"shape: {len(df_final)}")

    else: 
        print("no data was collected")

print("\n Finished!")

Season 2026 already have the folloing GP's [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

 Finished!


In [11]:
import polars as pl

querry = pl.scan_parquet(r'C:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\notebooks\Data\Raw')

querry.sink_csv('race_results.csv')

In [ ]:
import pandas as pd
from pathlib import Path 
import pyarrow.parquet as pq


dir = Path(r'C:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\notebooks\Data\Raw')
arquivos = sorted(dir.glob("*.parquet"))
saida = "race_results.parquet"

writer = None
try:
    for file in arquivos:
        parquet_file = pq.ParquetFile(file)


        if writer is None:
            writer = pq.ParquetWriter(saida, parquet_file.schema_arrow, compression="SNAPPY")

        for batch in parquet_file.iter_batches():
            writer.write_batch(batch)
finally:
    if writer:
        writer.close()


In [ ]:
import polars as pl 

querry = pl.scan_parquet('\notebooks\Data\Raw')
querry.sink_csv('race_results.csv')

Feature Engineering

In [20]:
import pandas as pd 

df = pd.read_parquet(r'C:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\notebooks\Data\Raw\result_2026.parquet')

print(df.shape)
print(df.head(10))

(418, 9)
   Season  Round       Date    Circuit                  Event Driver  \
0    2026      1 2026-03-08  Melbourne  Australian Grand Prix    RUS   
1    2026      1 2026-03-08  Melbourne  Australian Grand Prix    ANT   
2    2026      1 2026-03-08  Melbourne  Australian Grand Prix    LEC   
3    2026      1 2026-03-08  Melbourne  Australian Grand Prix    HAM   
4    2026      1 2026-03-08  Melbourne  Australian Grand Prix    NOR   
5    2026      1 2026-03-08  Melbourne  Australian Grand Prix    VER   
6    2026      1 2026-03-08  Melbourne  Australian Grand Prix    BEA   
7    2026      1 2026-03-08  Melbourne  Australian Grand Prix    LIN   
8    2026      1 2026-03-08  Melbourne  Australian Grand Prix    BOR   
9    2026      1 2026-03-08  Melbourne  Australian Grand Prix    GAS   

             Team  GridPosition  Position  
0        Mercedes           1.0       1.0  
1        Mercedes           2.0       2.0  
2         Ferrari           4.0       3.0  
3         Ferrari     

Temporal Separation

In [ ]:

import pandas as pd
import numpy as np

# Carregar resultados históricos
df = pd.read_parquet("data/raw/race_results.parquet")

df = df.sort_values(
    ["Date", "Season", "Round"]
).reset_index(drop=True)

# Identificador único de cada GP
df["RaceID"] = (
    df["Season"].astype(str)
    + "_"
    + df["Round"].astype(str)
)

# Variável alvo: conquistou pódio?
df["Podium"] = (
    df["Position"].notna()
    & df["Position"].between(1, 3)
).astype(int)

# 1. Histórico individual do piloto
driver_groups = df.groupby("Driver", sort=False)

df["DriverPodiumRateLast5"] = driver_groups[
    "Podium"
].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

df["DriverAvgPositionLast5"] = driver_groups[
    "Position"
].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# 2. Histórico da equipe
# Primeiro: uma linha por equipe em cada GP
team_races = (
    df.groupby(
        ["Season", "Round", "RaceID", "Date", "Team"],
        as_index=False
    )
    .agg(TeamPodium=("Podium", "max"))
    .sort_values(["Date", "Season", "Round"])
)

team_races["TeamPodiumRateLast5"] = (
    team_races.groupby("Team", sort=False)["TeamPodium"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

# Incorporar a feature no dataset de pilotos
df = df.merge(
    team_races[
        ["RaceID", "Team", "TeamPodiumRateLast5"]
    ],
    on=["RaceID", "Team"],
    how="left",
    validate="many_to_one"
)

# 3. Histórico do piloto no circuito
df["CircuitPodiumHistory"] = (
    df.groupby(
        ["Driver", "Circuit"],
        sort=False
    )["Podium"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=1).mean()
    )
)

# Preencher histórico inexistente
# O piloto não possui histórico anterior naquele contexto
df["DriverPodiumRateLast5"] = (
    df["DriverPodiumRateLast5"].fillna(0)
)

df["TeamPodiumRateLast5"] = (
    df["TeamPodiumRateLast5"].fillna(0)
)

df["CircuitPodiumHistory"] = (
    df["CircuitPodiumHistory"].fillna(0)
)

# Salvar dataset final
df.to_parquet(
    "data/processed/f1_features.parquet",
    index=False
)

print("Feature Engineering concluído!")
print(df.head())

print("\nFeatures disponíveis:")
print(df.columns.tolist())


In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
import pandas as pd

df = pd.read_parquet(
    "data/processed/f1_features.parquet"
)

features = [
    "GridPosition",
    "Driver",
    "Team",
    "Circuit",
    "DriverPodiumRateLast5",
    "DriverAvgPositionLast5",
    "TeamPodiumRateLast5",
    "CircuitPodiumHistory"
]

target = "Podium"

# Separação temporal: treino até 2023
train = df[df["Season"] <= 2023].copy()

# Teste: 2024 e 2025
test = df[df["Season"].isin([2024, 2025])].copy()

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

categorical_features = [
    "Driver",
    "Team",
    "Circuit"
]

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=100,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=100
)

predictions = model.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    (predictions >= 0.5).astype(int)
))